# Limpieza Tabla t_customer

In [0]:
%sql
-- Remover duplicados y añadir llave sustituta
use dw01.silver;

CREATE OR REPLACE TEMPORARY VIEW v_customers AS
WITH deduped AS (
    SELECT DISTINCT
        Customer_Name,
        Street,
        City,
        State,
        Customer_Type,
        Account_Manager 
    FROM dw01.bronze.t_customers
),
add_row AS (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY Customer_Name) AS customer_id,
        *
    FROM deduped
)
SELECT * 
FROM add_row;

create table if not exists t_customers FROM v_customers;
SELECT * from t_customers;

In [0]:
%sql
USE dw01.silver;

MERGE INTO t_customers AS target
USING (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY Customer_Name) AS customer_id,
        Customer_Name, Street, City, State, Customer_Type, Account_Manager
    FROM dw01.bronze.t_customers
) AS source
ON COALESCE(target.Customer_Name,'') = COALESCE(source.Customer_Name,'')
   AND COALESCE(target.Street,'') = COALESCE(source.Street,'')
   AND COALESCE(target.City,'') = COALESCE(source.City,'')
   AND COALESCE(target.State,'') = COALESCE(source.State,'')
   AND COALESCE(target.Customer_Type,'') = COALESCE(source.Customer_Type,'')
   AND COALESCE(target.Account_Manager,'') = COALESCE(source.Account_Manager,'')

WHEN MATCHED AND (
       COALESCE(target.Customer_Name,'') <> COALESCE(source.Customer_Name,'')
    OR COALESCE(target.Street,'') <> COALESCE(source.Street,'')
    OR COALESCE(target.City,'') <> COALESCE(source.City,'')
    OR COALESCE(target.State,'') <> COALESCE(source.State,'')
    OR COALESCE(target.Customer_Type,'') <> COALESCE(source.Customer_Type,'')
    OR COALESCE(target.Account_Manager,'') <> COALESCE(source.Account_Manager,'')
) THEN
  UPDATE SET
    target.Customer_Name = source.Customer_Name,
    target.Street = source.Street,
    target.City = source.City,
    target.State = source.State,
    target.Customer_Type = source.Customer_Type,
    target.Account_Manager = source.Account_Manager

WHEN NOT MATCHED THEN
  INSERT (customer_id, Customer_Name, Street, City, State, Customer_Type, Account_Manager)
  VALUES (source.customer_id, source.Customer_Name, source.Street, source.City, source.State, source.Customer_Type, source.Account_Manager);


# Limpieza Tabla Productos


In [0]:
%sql

use dw01.silver;

CREATE OR REPLACE TEMPORARY VIEW v_products as
WITH deduped_product AS (
    SELECT DISTINCT
        REPLACE(TRIM(BOTH '"' FROM Product_Name), '""', '"') AS Product_Name, 
        Product_Category, 
        Product_Container 
    FROM dw01.bronze.t_products
),
add_row_product AS (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY Product_Name) AS product_id,
        *
    FROM deduped_product
)
SELECT * 
FROM add_row_product;

create table if not exists t_products AS
SELECT * 
FROM v_products;

SELECT * from t_products;


In [0]:
%sql
USE dw01.silver;

-- Step 1: Get the max product_id from the target table
WITH max_id AS (
    SELECT COALESCE(MAX(product_id), 0) AS max_product_id
    FROM t_products
),
deduped_product AS (
    SELECT DISTINCT
        REPLACE(TRIM(BOTH '"' FROM Product_Name), '""', '"') AS Product_Name,
        Product_Category,
        Product_Container
    FROM dw01.bronze.t_products
),
source_with_id AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY Product_Name) + (SELECT max_product_id FROM max_id) AS product_id,
        Product_Name,
        Product_Category,
        Product_Container
    FROM deduped_product
)
MERGE INTO t_products AS target
USING source_with_id AS source
ON target.Product_Name = source.Product_Name
   AND target.Product_Category = source.Product_Category
   AND target.Product_Container = source.Product_Container
WHEN NOT MATCHED THEN
    INSERT (product_id, Product_Name, Product_Category, Product_Container)
    VALUES (source.product_id, source.Product_Name, source.Product_Category, source.Product_Container);


# Limpieza Tabla ShippingDetails

In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW v_shipping AS
WITH deduped_shipping AS (
    SELECT DISTINCT
        Ship_Mode
    FROM dw01.bronze.t_shippingdetails
), add_row_shipping AS (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY Ship_Mode) AS shipping_id,
        *
    FROM deduped_shipping
)
SELECT * 
FROM add_row_shipping;

create table if not exists t_shippingdetails AS
SELECT * 
FROM v_shipping;

SELECT * FROM t_shippingdetails



In [0]:
%sql
-- Step 1: Get the max shipping_id currently in the table

USE dw01.silver;

CREATE OR REPLACE TEMP VIEW v_max_shipping_id AS
SELECT COALESCE(MAX(shipping_id), 0) AS max_id
FROM t_shippingdetails;

-- Step 2: Prepare only new shipping modes
CREATE OR REPLACE TEMP VIEW v_new_shipping AS
WITH deduped_shipping AS (
    SELECT DISTINCT Ship_Mode
    FROM dw01.bronze.t_shippingdetails
),
existing_shipping AS (
    SELECT Ship_Mode
    FROM t_shippingdetails
),
new_only AS (
    SELECT d.Ship_Mode
    FROM deduped_shipping d
    LEFT JOIN existing_shipping e
    ON d.Ship_Mode = e.Ship_Mode
    WHERE e.Ship_Mode IS NULL
),
add_row_shipping AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY Ship_Mode) + (SELECT max_id FROM v_max_shipping_id) AS shipping_id,
        Ship_Mode
    FROM new_only
)
SELECT *
FROM add_row_shipping;

-- Step 3: Merge new shipping modes into the target table
MERGE INTO t_shippingdetails AS target
USING v_new_shipping AS source
ON target.Ship_Mode = source.Ship_Mode
WHEN NOT MATCHED THEN
  INSERT (shipping_id, Ship_Mode)
  VALUES (source.shipping_id, source.Ship_Mode);


# Limpieza Tabla Orders

In [0]:
%sql
USE dw01.silver;

create table if not exists t_orders AS
WITH deduped_orders AS (
    SELECT DISTINCT
        Order_No,
        Order_Date,
        Customer_Name,
        Address,
        City,
        State,
        Customer_Type,
        Account_Manager,
        Order_Priority,
        Product_Name,
        Product_Category,
        Product_Container,
        Ship_Mode,
        Ship_Date,
        TRY_CAST(REGEXP_REPLACE(Cost_Price,'[$,]','') AS DECIMAL(10,2)) AS Cost_Price,
        TRY_CAST(REGEXP_REPLACE(Retail_Price,'[$,]','') AS DECIMAL(10,2)) AS Retail_Price,
        TRY_CAST(REGEXP_REPLACE(Profit_Margin,'[$,]','') AS DECIMAL(10,2)) AS Profit_Margin,
        TRY_CAST(Order_Quantity AS INT) AS Order_Quantity,
        TRY_CAST(REGEXP_REPLACE(Sub_Total,'[$,]','') AS DECIMAL(12,2)) AS Sub_Total,
        TRY_CAST(REGEXP_REPLACE(Discount_Amount,'[$,]','') AS DECIMAL(12,2)) AS Discount_Amount,
        TRY_CAST(REGEXP_REPLACE(Order_Total,'[$,]','') AS DECIMAL(12,2)) AS Order_Total,
        TRY_CAST(REGEXP_REPLACE(Shipping_Cost,'[$,]','') AS DECIMAL(10,2)) AS Shipping_Cost,
        TRY_CAST(REGEXP_REPLACE(Total,'[$,]','') AS DECIMAL(12,2)) AS Total
    FROM dw01.bronze.t_orders
),

adding_order_id AS (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY Order_No) AS order_id,
        *
    FROM deduped_orders
)

SELECT *
FROM adding_order_id;


In [0]:
%sql
USE dw01.silver;

MERGE INTO t_orders AS target
USING (
    SELECT DISTINCT
        *,
        ROW_NUMBER() OVER (ORDER BY Order_No) AS rn
    FROM (
        SELECT
            Order_No,
            Order_Date,
            Customer_Name,
            Address,
            City,
            State,
            Customer_Type,
            Account_Manager,
            Order_Priority,
            Product_Name,
            Product_Category,
            Product_Container,
            Ship_Mode,
            Ship_Date,
            TRY_CAST(REGEXP_REPLACE(Cost_Price,'[$,]','') AS DECIMAL(10,2)) AS Cost_Price,
            TRY_CAST(REGEXP_REPLACE(Retail_Price,'[$,]','') AS DECIMAL(10,2)) AS Retail_Price,
            TRY_CAST(REGEXP_REPLACE(Profit_Margin,'[$,]','') AS DECIMAL(10,2)) AS Profit_Margin,
            TRY_CAST(Order_Quantity AS INT) AS Order_Quantity,
            TRY_CAST(REGEXP_REPLACE(Sub_Total,'[$,]','') AS DECIMAL(12,2)) AS Sub_Total,
            TRY_CAST(REGEXP_REPLACE(Discount_Amount,'[$,]','') AS DECIMAL(12,2)) AS Discount_Amount,
            TRY_CAST(REGEXP_REPLACE(Order_Total,'[$,]','') AS DECIMAL(12,2)) AS Order_Total,
            TRY_CAST(REGEXP_REPLACE(Shipping_Cost,'[$,]','') AS DECIMAL(10,2)) AS Shipping_Cost,
            TRY_CAST(REGEXP_REPLACE(Total,'[$,]','') AS DECIMAL(12,2)) AS Total
        FROM dw01.bronze.t_orders
    ) AS deduped_orders
) AS source
ON target.Order_No = source.Order_No
   AND target.Order_Date = source.Order_Date
   AND target.Customer_Name = source.Customer_Name
   AND (target.Address = source.Address OR (target.Address IS NULL AND source.Address IS NULL))
   AND (target.City = source.City OR (target.City IS NULL AND source.City IS NULL))
   AND (target.State = source.State OR (target.State IS NULL AND source.State IS NULL))
   AND (target.Customer_Type = source.Customer_Type OR (target.Customer_Type IS NULL AND source.Customer_Type IS NULL))
   AND (target.Account_Manager = source.Account_Manager OR (target.Account_Manager IS NULL AND source.Account_Manager IS NULL))
   AND (target.Order_Priority = source.Order_Priority OR (target.Order_Priority IS NULL AND source.Order_Priority IS NULL))
   AND (target.Product_Name = source.Product_Name OR (target.Product_Name IS NULL AND source.Product_Name IS NULL))
   AND (target.Product_Category = source.Product_Category OR (target.Product_Category IS NULL AND source.Product_Category IS NULL))
   AND (target.Product_Container = source.Product_Container OR (target.Product_Container IS NULL AND source.Product_Container IS NULL))
   AND (target.Ship_Mode = source.Ship_Mode OR (target.Ship_Mode IS NULL AND source.Ship_Mode IS NULL))
   AND (target.Ship_Date = source.Ship_Date OR (target.Ship_Date IS NULL AND source.Ship_Date IS NULL))
   AND (target.Cost_Price = source.Cost_Price OR (target.Cost_Price IS NULL AND source.Cost_Price IS NULL))
   AND (target.Retail_Price = source.Retail_Price OR (target.Retail_Price IS NULL AND source.Retail_Price IS NULL))
   AND (target.Profit_Margin = source.Profit_Margin OR (target.Profit_Margin IS NULL AND source.Profit_Margin IS NULL))
   AND (target.Order_Quantity = source.Order_Quantity OR (target.Order_Quantity IS NULL AND source.Order_Quantity IS NULL))
   AND (target.Sub_Total = source.Sub_Total OR (target.Sub_Total IS NULL AND source.Sub_Total IS NULL))
   AND (target.Discount_Amount = source.Discount_Amount OR (target.Discount_Amount IS NULL AND source.Discount_Amount IS NULL))
   AND (target.Order_Total = source.Order_Total OR (target.Order_Total IS NULL AND source.Order_Total IS NULL))
   AND (target.Shipping_Cost = source.Shipping_Cost OR (target.Shipping_Cost IS NULL AND source.Shipping_Cost IS NULL))
   AND (target.Total = source.Total OR (target.Total IS NULL AND source.Total IS NULL))
WHEN NOT MATCHED THEN
  INSERT (
      Order_No,
      Order_Date,
      Customer_Name,
      Address,
      City,
      State,
      Customer_Type,
      Account_Manager,
      Order_Priority,
      Product_Name,
      Product_Category,
      Product_Container,
      Ship_Mode,
      Ship_Date,
      Cost_Price,
      Retail_Price,
      Profit_Margin,
      Order_Quantity,
      Sub_Total,
      Discount_Amount,
      Order_Total,
      Shipping_Cost,
      Total
  )
  VALUES (
      source.Order_No,
      source.Order_Date,
      source.Customer_Name,
      source.Address,
      source.City,
      source.State,
      source.Customer_Type,
      source.Account_Manager,
      source.Order_Priority,
      source.Product_Name,
      source.Product_Category,
      source.Product_Container,
      source.Ship_Mode,
      source.Ship_Date,
      source.Cost_Price,
      source.Retail_Price,
      source.Profit_Margin,
      source.Order_Quantity,
      source.Sub_Total,
      source.Discount_Amount,
      source.Order_Total,
      source.Shipping_Cost,
      source.Total
  );


# Creacion tabla fecha

In [0]:
%sql
-- Remove duplicates, add surrogate key, and convert dates to DATE
create table if not exists t_dates as  
WITH deduped_date AS (
    SELECT DISTINCT
        Order_Date, 
        Ship_Date
    FROM dw01.bronze.t_orders
),
date_union AS (
    SELECT Order_Date AS date FROM deduped_date
    UNION
    SELECT Ship_Date AS date FROM deduped_date
),
add_row_date AS (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY date) AS date_id,
        date
    FROM date_union
),
date_table AS (
    SELECT 
        date_id,
        TO_DATE(date, 'dd-MM-yyyy') AS date
    FROM add_row_date
)
SELECT
    date_id,
    date,
    EXTRACT(DAY FROM date) AS day,
    EXTRACT(MONTH FROM date) AS month,
    EXTRACT(YEAR FROM date) AS year
FROM date_table;


In [0]:
%sql
-- Incremental merge into t_dates
WITH deduped_date AS (
    SELECT DISTINCT Order_Date, Ship_Date
    FROM dw01.bronze.t_orders
),
date_union AS (
    SELECT Order_Date AS date FROM deduped_date
    UNION
    SELECT Ship_Date AS date FROM deduped_date
),
-- Assign row numbers for new dates
new_dates_with_id AS (
    SELECT 
        COALESCE((SELECT MAX(date_id) FROM t_dates), 0) 
          + ROW_NUMBER() OVER (ORDER BY du.date) AS date_id,
        TO_DATE(du.date, 'dd-MM-yyyy') AS date,
        EXTRACT(DAY FROM TO_DATE(du.date, 'dd-MM-yyyy')) AS day,
        EXTRACT(MONTH FROM TO_DATE(du.date, 'dd-MM-yyyy')) AS month,
        EXTRACT(YEAR FROM TO_DATE(du.date, 'dd-MM-yyyy')) AS year
    FROM date_union du
    LEFT JOIN t_dates td
      ON TO_DATE(du.date, 'dd-MM-yyyy') = td.date
    WHERE td.date IS NULL
)
MERGE INTO t_dates AS target
USING new_dates_with_id AS source
ON target.date = source.date
WHEN NOT MATCHED THEN
INSERT (date_id, date, day, month, year)
VALUES (source.date_id, source.date, source.day, source.month, source.year);
